# Transformer LM Trainer — Hindi / Nepali (20k BPE)

All bugs (Memory overflow, Causal Mask/Perplexity cheating, and W&B logging) are fixed here.

**Before running:** Turn on **GPU** under *Settings → Accelerator*, and add your `HF_TOKEN` and (optionally)
`WANDB_API_KEY` under *Add-ons → Secrets*.

## 1. Install dependencies

In [ ]:
!pip install -q wandb huggingface_hub datasets

## 2. Project-specific tokenizer module

Point this to your custom `BPETokenizer` class from `tokenizer/tokenizers.py` if needed.

In [ ]:
import sys
# sys.path.append("/kaggle/working/lma-project")

## 3. Imports

In [ ]:
import math
import os
import time
import random
import json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import wandb
from huggingface_hub import HfApi, hf_hub_download

In [ ]:
# TOKENIZER CODE 
import re
import unicodedata
import pickle
import heapq
from collections import defaultdict, Counter

random.seed(69)
BPE_MERGES = 20000 

class Tokenizer:
    def train(self, data): pass
    def tokenize(self, text): raise NotImplementedError
    def save(self, path): pass
    def load(self, path): pass

class RegexTokenizer(Tokenizer):
    def __init__(self):
        self.pattern = re.compile(r"[\w\u0900-\u097F]+|[^\w\s]", re.UNICODE)
    def tokenize(self, text):
        return self.pattern.findall(text)

class customBPE(Tokenizer):
    def __init__(self, num_merges=BPE_MERGES, min_pair_frequency=2):
        self.num_merges = num_merges
        self.min_pair_frequency = min_pair_frequency
        self.merges = {}
        self.vocab = {}
        self._token_cache = {}
        self.pre_tokenizer = RegexTokenizer()

    def _bpe_word(self, word):
        cached = self._token_cache.get(word)
        if cached is not None: return cached
        if not hasattr(self, 'merge_ranks'):
            self.merge_ranks = {pair: idx for idx, pair in enumerate(self.merges.keys())}
        tokens = list(word) + ["</w>"]
        while len(tokens) > 1:
            pairs = [(tokens[i], tokens[i + 1]) for i in range(len(tokens) - 1)]
            best_pair = None
            earliest_rank = float('inf')
            for pair in pairs:
                rank = self.merge_ranks.get(pair)
                if rank is not None and rank < earliest_rank:
                    earliest_rank = rank
                    best_pair = pair
            if best_pair is None: break
            left, right = best_pair
            merged_tokens = []
            i = 0
            n = len(tokens)
            while i < n:
                if i < n - 1 and tokens[i] == left and tokens[i + 1] == right:
                    merged_tokens.append(left + right)
                    i += 2
                else:
                    merged_tokens.append(tokens[i])
                    i += 1
            tokens = merged_tokens
        cleaned_tokens = [t.replace('</w>', '') for t in tokens if t != '</w>']
        cleaned_tokens = [token for token in cleaned_tokens if token]
        self._token_cache[word] = cleaned_tokens
        return cleaned_tokens

    def tokenize(self, text):
        words = self.pre_tokenizer.tokenize(text)
        result = []
        for word in words: result.extend(self._bpe_word(word))
        return result

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode(self, text):
        if text in ("<S>", "</S>", "<pad>", "<unk>") and text in self.vocab:
            return [self.vocab[text]]
        unk_id = self.vocab.get("<unk>", 0)
        boundary_id = self.vocab.get("</w>", unk_id)
        words = self.pre_tokenizer.tokenize(text)
        ids = []
        for word in words:
            for tok in self._bpe_word(word):
                ids.append(self.vocab.get(tok, unk_id))
            ids.append(boundary_id)
        return ids

    def decode(self, ids):
        if not hasattr(self, '_id_to_token') or len(self._id_to_token) != len(self.vocab):
            self._id_to_token = {i: t for t, i in self.vocab.items()}
        pieces = []
        for i in ids:
            tok = self._id_to_token.get(i, "<unk>")
            if tok == "</w>": pieces.append(" ")
            elif tok in ("<S>", "</S>", "<pad>"): continue
            else: pieces.append(tok)
        return "".join(pieces).strip()

    def load(self, path):
        with open(path, 'rb') as f:
            model = pickle.load(f)
        if isinstance(model, dict) and "merges" in model:
            self.num_merges = model.get("num_merges", self.num_merges)
            self.min_pair_frequency = model.get("min_pair_frequency", self.min_pair_frequency)
            self.merges = model["merges"]
            self.vocab = model.get("vocab", {})
        else:
            self.merges = model
            self.vocab = {}
        self.merge_ranks = {pair: idx for idx, pair in enumerate(self.merges.keys())}
        self._token_cache.clear()

## 4. Credentials (Kaggle Secrets)

In [ ]:
HF_TOKEN = None
WANDB_API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try: HF_TOKEN = _secrets.get_secret("HF_TOKEN")
    except Exception: print("No HF_TOKEN secret found.")
    try: WANDB_API_KEY = _secrets.get_secret("WANDB_API_KEY")
    except Exception: print("No WANDB_API_KEY secret found.")
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is not set.")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
else:
    os.environ.setdefault("WANDB_MODE", "offline")

## 5. Config

In [ ]:
# --- Edit these instead of passing CLI args ---
LANG = "hi"
MAX_STEPS = 50000
RESUME_FROM_HF = False

WANDB_PROJECT = "lma-project"
WANDB_ENTITY = "bropal-iiit-hyderabad"
HF_REPO_ID = f"bropal/lma-project"
TOKENIZER_REPO_ID = f"bropal/lma-project"

MERGES = 20000
CHECKPOINT_STEPS = 5000
LOG_STEPS = 200
EVAL_STEPS = 1000
SEQ_LEN = 256
BATCH_SIZE = 8
LR = 3e-4

## 6. Model (FIXED: Causal Mask Added)

In [ ]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=384, n_layers=6, n_heads=6,
                 d_ff=1536, context_length=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.context_length = context_length

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(context_length, d_model)

        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
                batch_first=True, dropout=dropout, activation="gelu"
            ) for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_embedding.weight

    def forward(self, input_ids, targets=None):
        B, T = input_ids.shape
        tok_emb = self.token_embedding(input_ids)
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        pos_emb = self.position_embedding(pos)
        x = tok_emb + pos_emb

        # --- BUG FIX: CAUSAL MASK ---
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=input_ids.device)

        for blk in self.blocks:
            x = blk(x, src_mask=causal_mask, is_causal=True)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            # --- BUG FIX: REMOVED DOUBLE SHIFT ---
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), 
                targets.view(-1), 
                ignore_index=-1
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0, top_k=40):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = input_ids if input_ids.size(1) <= self.context_length else input_ids[:, -self.context_length:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, idx_next], dim=1)

        return input_ids

## 7. Dataset (FIXED: Offset alignment)

In [ ]:
class TextDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        self.token_ids = token_ids
        self.seq_len = seq_len
        self.len = max(0, len(token_ids) - seq_len - 1)

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        chunk = self.token_ids[idx:idx + self.seq_len + 1]
        if hasattr(chunk, 'astype'):
            chunk = chunk.astype(np.int64)
        
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

## 8. Utility functions (FIXED: Memory & Timeout)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def compute_perplexity(model, dataloader, device, max_batches=200):
    model.eval()
    total_loss = 0.0
    n_seqs = 0
    with torch.no_grad():
        for i, (x, y) in enumerate(dataloader):
            if i >= max_batches: break
            x, y = x.to(device), y.to(device)
            _, loss = model(x, y)
            if loss is not None:
                total_loss += loss.item()
                n_seqs += 1
    if n_seqs == 0: return float("inf")
    return math.exp(total_loss / n_seqs)

def load_tokenizer_from_hf(repo_id, token):
    api = HfApi(token=token)
    tokenizer_filename = None
    files = api.list_repo_files(repo_id=repo_id, token=token)
    pkl_files = [f for f in files if f.endswith(".pkl")]
    if pkl_files:
        for pf in pkl_files:
            if "20000" in pf or "artifacts_20000" in pf:
                tokenizer_filename = pf
                break
        if tokenizer_filename is None:
            tokenizer_filename = pkl_files[0]
    if tokenizer_filename is None:
        raise FileNotFoundError(f"No 20k tokenizer .pkl found in {repo_id}.")

    print(f"Downloading tokenizer from {repo_id}:{tokenizer_filename}...")
    local_path = hf_hub_download(repo_id=repo_id, filename=tokenizer_filename, token=token)
    tokenizer = customBPE()
    tokenizer.load(local_path)
    return tokenizer

def load_tokenized_data_from_hf(data_repo_id, token, split="train"):
    from datasets import load_dataset
    try:
        dataset = load_dataset(data_repo_id, split=split, token=token)
        MAX_TOKENS = 12_000_000 if split == "train" else 500_000
        print(f"Extracting up to {MAX_TOKENS} tokens for {split} split...")
        
        tokens_list = []
        for row in dataset:
            if isinstance(row["input_ids"], (list, tuple)):
                tokens_list.extend(row["input_ids"])
            else:
                tokens_list.append(row["input_ids"])
            if len(tokens_list) >= MAX_TOKENS:
                break
                
        token_ids = np.array(tokens_list[:MAX_TOKENS], dtype=np.uint16)
        print(f"Loaded {split} split: {len(token_ids)} tokens (Memory Safe!)")
        return token_ids
    except Exception as e:
        print(f"Could not load '{split}' split from {data_repo_id}: {e}")
        return None

def upload_to_hf(local_path, repo_id, hf_token, path_in_repo):
    api = HfApi(token=hf_token)
    try:
        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=path_in_repo,
            repo_id=repo_id,
            token=hf_token,
        )
        print(f"  Uploaded: {path_in_repo}")
        return True
    except Exception as e:
        print(f"  Warning: Failed to upload {path_in_repo}: {e}")
        return False

## 9. Initialize Weights & Biases (RESTORED)

In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=f"train_{LANG}_bpe{MERGES}_{int(time.time())}",
    reinit=True,
    config={
        "lang": LANG,
        "merges": MERGES,
        "max_steps": MAX_STEPS,
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "seq_len": SEQ_LEN,
    },
)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 10. Load tokenizer and data

In [ ]:
print(f"Loading 20k tokenizer for {LANG} from HF Hub...")
tokenizer = load_tokenizer_from_hf(TOKENIZER_REPO_ID, HF_TOKEN)
vocab_size = tokenizer.vocab_size
print(f"Vocab size: {vocab_size}")

print(f"Loading tokenized data for {LANG} from HF Hub...")
train_token_ids = load_tokenized_data_from_hf(TOKENIZER_REPO_ID, HF_TOKEN, "train")
val_token_ids = load_tokenized_data_from_hf(TOKENIZER_REPO_ID, HF_TOKEN, "validation")

if train_token_ids is not None:
    train_dataset_obj = TextDataset(train_token_ids, SEQ_LEN)
    val_dataset_obj = TextDataset(val_token_ids, SEQ_LEN) if val_token_ids is not None else None
else:
    print("Falling back to synthetic random data (smoke test)")
    train_token_ids = np.random.randint(0, vocab_size, 5000 * SEQ_LEN, dtype=np.uint16)
    val_token_ids = np.random.randint(0, vocab_size, 1000 * SEQ_LEN, dtype=np.uint16)
    train_dataset_obj = TextDataset(train_token_ids, SEQ_LEN)
    val_dataset_obj = TextDataset(val_token_ids, SEQ_LEN)

# train_loader is completely removed to prevent system RAM shuffle crashes.
val_loader = DataLoader(val_dataset_obj, batch_size=BATCH_SIZE, shuffle=False) if val_dataset_obj else None

## 11. Build model, optimizer, and (optionally) resume

In [ ]:
model = TransformerLM(vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

start_step = 0
if RESUME_FROM_HF:
    print("Attempting to resume from latest HF checkpoint...")
    api = HfApi(token=HF_TOKEN)
    try:
        files = api.list_repo_files(repo_id=HF_REPO_ID, token=HF_TOKEN)
        ckpt_files = [f for f in files if f.startswith("checkpoints/") and f.endswith(".pt")]
        step_nums = []
        for f in ckpt_files:
            try:
                step = int(f.split("_")[-1].split(".")[0])
                step_nums.append((step, f))
            except ValueError:
                pass
        if step_nums:
            step_nums.sort(reverse=True)
            latest_ckpt = step_nums[0][1]
            print(f"Downloading checkpoint: {latest_ckpt}")
            ckpt_path = hf_hub_download(repo_id=HF_REPO_ID, filename=latest_ckpt, token=HF_TOKEN)
            checkpoint = torch.load(ckpt_path, map_location=device)
            model.load_state_dict(checkpoint["model_state"])
            optimizer.load_state_dict(checkpoint["optimizer_state"])
            start_step = checkpoint["step"] + 1
            print(f"Resuming from step {start_step}")
        else:
            print("No checkpoints found - starting from scratch.")
    except Exception as e:
        print(f"Could not resume from HF: {e}")

## 12. Training loop

In [ ]:
global_step = start_step
pbar = tqdm(total=MAX_STEPS, desc="Training", initial=start_step, unit="step")

model.train()
last_loss = 0.0

while global_step < MAX_STEPS:
    
    # Memory safe random batch sampling (0 System RAM cost)
    ix = torch.randint(0, len(train_dataset_obj), (BATCH_SIZE,))
    x = torch.stack([train_dataset_obj[i.item()][0] for i in ix]).to(device)
    y = torch.stack([train_dataset_obj[i.item()][1] for i in ix]).to(device)

    logits, loss = model(x, y)
    last_loss = loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if global_step % LOG_STEPS == 0:
        wandb.log({
            "loss": last_loss,
            "lr": optimizer.param_groups[0]["lr"],
            "step": global_step,
            "perplexity": math.exp(last_loss) if last_loss < 10 else float("inf"),
        })

    pbar.update(1)
    if global_step % 100 == 0:
        pbar.set_postfix({"loss": f"{last_loss:.4f}", "ppl": f"{math.exp(min(last_loss, 20)):.2f}"})

    if global_step > 0 and global_step % CHECKPOINT_STEPS == 0:
        ckpt_name = f"checkpoint_step_{global_step}.pt"
        ckpt_path = f"/kaggle/working/{ckpt_name}"
        torch.save({
            "step": global_step,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "loss": last_loss,
            "vocab_size": vocab_size,
            "lang": LANG,
        }, ckpt_path)
        upload_to_hf(ckpt_path, HF_REPO_ID, HF_TOKEN, f"checkpoints/{ckpt_name}")
        print(f"\nSaved checkpoint at step {global_step}")
        wandb.log({"checkpoint_step": global_step})

    if global_step > 0 and global_step % EVAL_STEPS == 0 and val_loader is not None:
        model.eval()
        ppl = compute_perplexity(model, val_loader, device)
        wandb.log({"perplexity": ppl, "step": global_step})
        print(f"\nStep {global_step}: Val Perplexity = {ppl:.2f}")
        model.train()

    global_step += 1

print(f"\nTraining completed, step {global_step}")
pbar.close()

## 13. Generate sample sentences

In [ ]:
print("\n" + "=" * 60)
print("Generating 5 sample sentences...")
print("=" * 60)

model.eval()
with torch.no_grad():
    for i in range(5):
        prompt = tokenizer.encode("<S>")  # start token
        if not prompt:
            prompt = [0]  # fallback to vocab token 0
        context = torch.tensor([prompt], dtype=torch.long).to(device)

        generated = model.generate(context, max_new_tokens=50, temperature=0.8, top_k=40)

        generated_ids = generated.squeeze(0).tolist()
        text = tokenizer.decode(generated_ids)
        print(f"\nSample {i + 1}: {text}")
        wandb.log({f"sample_{i + 1}": text})

## 14. Save final model + tokenizer to the HF Hub

In [ ]:
print("\nSaving model and tokenizer to HF Hub...")

final_ckpt_path = "/kaggle/working/model_final.pt"
torch.save({
    "step": global_step,
    "model_state": model.state_dict(),
    "vocab_size": vocab_size,
    "lang": LANG,
}, final_ckpt_path)
upload_to_hf(final_ckpt_path, HF_REPO_ID, HF_TOKEN, "model_final.pt")

config_path = "/kaggle/working/config.json"
with open(config_path, "w") as f:
    json.dump({
        "vocab_size": vocab_size,
        "d_model": 384,
        "n_layers": 6,
        "n_heads": 6,
        "d_ff": 1536,
        "context_length": SEQ_LEN,
        "lang": LANG,
        "merges": MERGES,
    }, f, indent=2)
upload_to_hf(config_path, HF_REPO_ID, HF_TOKEN, "config.json")

tokenizer_local_path = "/kaggle/working/tokenizer.pkl"
try:
    tokenizer.save(tokenizer_local_path)
    upload_to_hf(tokenizer_local_path, TOKENIZER_REPO_ID, HF_TOKEN, "tokenizer.pkl")
except AttributeError:
    print("Tokenizer object has no .save(path) method - skipping tokenizer upload.")

wandb.log({
    "final_step": global_step,
    "final_loss": last_loss,
    "merges": MERGES,
    "lang": LANG,
})
wandb.finish()

print("\nTraining complete!")
print(f"Model pushed to: {HF_REPO_ID}")
print(f"Tokenizer pushed to: {TOKENIZER_REPO_ID}")
print(f"Final step: {global_step}")